## Datasets and research question

I am looking at two sets of estimated ages: van den Berg (55 rows) and Krause (61 rows). I want to find out whether any clusters have unusual ages compared with other clusters of similar metallicity.

### Fields I need 

`Age`: estimated age of a cluster, in billions of years
`FeH`: metallicity. It compares the amount of iron with hydrogen; higher value (for example -0.8 instead of -2.0) means more iron relative to hydrogen. It is a relative field, it has no unit.
- `Age_err`: the age error listed by van den Berg, in billions of years. Krause does not have this field.
- '#NGC' and 'Name' in van den Berg and 'Object' in Krause: names or numbers used to identify clusters and compare it across files.

I checked the row counts, column names and example rows. Neither file has missing values in 'Age' or 'FeH'. Some van den Berg `Name` entries are `XXXX`, so I cannot rely on that column alone when matching clusters.

### stuff I noticed on a first look 

The ages spread more widely in Krause than in van den Berg. Pal 12 is young in both files: 9.0 billion in van den Berg and 8.6 in Krause. Some ages differ a lot between files: NGC 1851 is 11.0 in van den Berg but 7.64 in Krause, while NGC 7099 is 13.0 versus 14.6. Terzan 7 is young in Krause (7.4) but I cannot find it in van den Berg.

These are points to check later, not final outliers. I need to compare ages at similar metallicities and also look carefully at the clusters where the two files disagree.

## Choosing which measurements to use

Krause has 61 rows, and every row is labelled 'GC' (globular cluster). Van den Berg has 55 rows. Neither file has missing 'Age' or 'FeH' values, so I can use all 61 and 55 rows for the first age-metallicity plots.

I will show van den Berg's values first cuz it also gives me 'Age_err', then show Krause's values separately for comparison. This does not mean van den Berg is automatically correct.

If the two files give different ages for the same cluster, I will keep both values and their source names visible. I will not average them. I will check the larger disagreements later before deciding how much confidence to put on outliers.

In [3]:
import re
import pandas as pd 

vanderberg = pd.read_csv("vandenBerg_table2.csv")
krause = pd.read_csv("Krause21.csv")
harris = pd.read_csv("HarrisPartI.csv")


def vdb_harris_id(row):
    # most van der berg has an NGC number such as 104
    ngc = str(row['#NGC']).strip()

    # Harris writes the same clusters as "NGC 104"
    if ngc.isdigit():
        return f"NGC {ngc}"

    # if the NGC says XXXX the we try the clusters other names, we only include names 
    # we checked
    checked_names = {
        "Pal12": "Pal 12",
        "Ter8" : "Terzan 8",
        "Arp21" : "Arp 2"
    }

    # if no name, we come back
    return checked_names.get(str(row["Name"]).strip())

def krause_harris_id(object_name):
    # Krause writes names like "NGC104" without space
    name = str(object_name).strip()
    ngc = re.fullmatch(r"NGC\s*(\d+)", name)

    # Turn NGC104 inot Harris "NGC 104"
    if ngc:
        return f"NGC {ngc.group(1)}"

    # These names are written differently in Krause and Harris so we gotta fix them
    checked_names = {
        "Ruprecht106": "Rup 106",
        "Terzan7": "Terzan 7",
        "Palomar12": "Pal 12",

    }

    # leave the rest unmatched

    return checked_names.get(name)

# Add a new matching ID to each age table
# original stuff is unchanged
vanderberg["harris_id"] = vanderberg.apply(vdb_harris_id, axis=1)
krause["harris_id"] = krause["Object"].map(krause_harris_id)

harris_ids = set(harris["ID"])

for source, data, original_id in [
    ("van den Berg", vanderberg, "Name"),
    ("Krause", krause, "Object"),
]:

    matched = data["harris_id"].isin(harris_ids)

    # seperate the done stuff from the useless one
    print(source, "matched:", matched.sum(), "out of", len(data))
    print("Unmatched rows:")
    print(data.loc[~matched, [original_id, "harris_id", "Age", "FeH"]])

    print("Repeated matching IDs:", data["harris_id"].dropna().duplicated().sum())

van den Berg matched: 55 out of 55
Unmatched rows:
Empty DataFrame
Columns: [Name, harris_id, Age, FeH]
Index: []
Repeated matching IDs: 0
Krause matched: 61 out of 61
Unmatched rows:
Empty DataFrame
Columns: [Object, harris_id, Age, FeH]
Index: []
Repeated matching IDs: 0
